# 4. Performance Profiling and Optimization

This notebook proves the execution speedup achieved via B-Tree indexing in SQLite and profiles the CPU execution of our Python ETL code.

In [ ]:
import sys
import time
import cProfile
import pstats
import io
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "src").exists():
    sys.path.insert(0, str(PROJECT_ROOT))
else:
    PROJECT_ROOT = PROJECT_ROOT.parent
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import get_db_connection, DB_PATH, DATA_DIR, PROFILING_DIR

PROFILING_DIR.mkdir(exist_ok=True)
conn = get_db_connection(DB_PATH)

## 4.1 SQLite Query Profiling (`EXPLAIN QUERY PLAN`)

In [ ]:
queries = {
    "Q1: Count reviews per year": """
        SELECT year, COUNT(*) AS cnt
        FROM reviews GROUP BY year ORDER BY year
    """,
    "Q2: Top 10 hotels by avg rating (min 50 reviews)": """
        SELECT hotel_id, AVG(rating_overall) AS avg_r, COUNT(*) AS cnt
        FROM reviews GROUP BY hotel_id HAVING cnt >= 50
        ORDER BY avg_r DESC LIMIT 10
    """,
    "Q3: Monthly trends": """
        SELECT year, month, AVG(rating_overall) AS avg_r, COUNT(*) AS cnt
        FROM reviews GROUP BY year, month ORDER BY year, month
    """,
    "Q4: Filter by hotel and date range": """
        SELECT * FROM reviews
        WHERE hotel_id = (SELECT hotel_id FROM hotels ORDER BY num_reviews DESC LIMIT 1)
          AND date_parsed BETWEEN '2011-01-01' AND '2012-12-31'
    """,
    "Q5: Join hotels and reviews": """
        SELECT h.hotel_id, h.num_reviews,
               AVG(r.rating_overall) AS avg_r
        FROM hotels h JOIN reviews r ON h.hotel_id = r.hotel_id
        GROUP BY h.hotel_id
        ORDER BY avg_r DESC LIMIT 20
    """,
}

In [ ]:
# 1. Profile Queries WITH Indexes
results_with = {}
for name, sql in queries.items():
    plan = conn.execute(f"EXPLAIN QUERY PLAN {sql}").fetchall()
    plan_str = "\n".join(str(row) for row in plan)

    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        conn.execute(sql).fetchall()
        times.append(time.perf_counter() - t0)
    avg_t = sum(times) / len(times)
    results_with[name] = {"time": avg_t, "plan": plan_str}

# 2. Drop Indexes Temporarily
idx_names = [row[0] for row in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='index' AND name LIKE 'idx_%'"
).fetchall()]
for idx in idx_names:
    conn.execute(f"DROP INDEX IF EXISTS {idx}")
conn.commit()
print(f"Dropped {len(idx_names)} indexes for testing...")

In [ ]:
# 3. Profile Queries WITHOUT Indexes
results_without = {}
for name, sql in queries.items():
    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        conn.execute(sql).fetchall()
        times.append(time.perf_counter() - t0)
    avg_t = sum(times) / len(times)
    results_without[name] = {"time": avg_t}

# 4. Restore Indexes
schema_sql = (DATA_DIR / "data_schema.sql").read_text()
for line in schema_sql.splitlines():
    if line.strip().upper().startswith("CREATE INDEX"):
        conn.execute(line)
conn.commit()
print("Indexes restored.")

In [ ]:
# 5. Display Comparisons
rows = []
for name in queries:
    t_with = results_with[name]["time"] * 1000
    t_without = results_without[name]["time"] * 1000
    speedup = t_without / t_with if t_with > 0 else 0
    rows.append({"Query": name, "With Index (ms)": round(t_with, 2),
                 "No Index (ms)": round(t_without, 2),
                 "Speedup": f"{speedup:.1f}x"})
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

# Save to file
with open(PROFILING_DIR / "query_results.txt", "w") as f:
    f.write("Query Profiling Results\n")
    f.write("=" * 60 + "\n")
    f.write(comparison.to_string(index=False))
    for name in queries:
        f.write(f"\n\n{name}\n")
        f.write(f"Plan (indexed):\n{results_with[name]['plan']}\n")

## 4.2 Python CPU Profiling (`cProfile`)

In [ ]:
from src.benchmarking import compute_hotel_features, cluster_hotels

pr = cProfile.Profile()
pr.enable()

# Profile the benchmarking pipeline (which is computationally intense)
features = compute_hotel_features(db_path=DB_PATH, min_reviews=10)
if not features.empty:
    df_cl, sil, _, _ = cluster_hotels(features, n_clusters=4)

pr.disable()

# Capture profiling output
s = io.StringIO()
ps = pstats.Stats(pr, stream=s)
ps.sort_stats("cumulative")
ps.print_stats(20)
profile_output = s.getvalue()
print(profile_output[:2000] + "\n...[truncated]")

with open(PROFILING_DIR / "code_profiling.txt", "w") as f:
    f.write("Code Profiling Results\n")
    f.write("=" * 60 + "\n")
    f.write("Profiled: compute_hotel_features() + cluster_hotels()\n")
    f.write(profile_output)